In [0]:
-- ============================================================
-- 30-DAY READMISSION ANALYSIS
-- Business question: of inpatient discharges, what share are
-- followed by another inpatient admission within 30 days?
-- Technique: LEAD() sees each patient's NEXT admission without
-- a self-join; the window handles per-patient ordering.
-- ============================================================
WITH inpatient_stays AS (
  SELECT
    patient_id,
    encounter_id,
    start_ts  AS admit_ts,
    stop_ts   AS discharge_ts
  FROM health_lakehouse.gold.fact_encounters
  WHERE encounter_class = 'inpatient'
    AND stop_ts IS NOT NULL
),
sequenced AS (
  SELECT
    patient_id,
    encounter_id,
    admit_ts,
    discharge_ts,
    -- the same patient's NEXT admission, chronologically
    LEAD(admit_ts) OVER (
      PARTITION BY patient_id
      ORDER BY admit_ts
    ) AS next_admit_ts
  FROM inpatient_stays
),
flagged AS (
  SELECT
    *,
    DATEDIFF(next_admit_ts, discharge_ts) AS days_to_next_admit,
    CASE
      WHEN next_admit_ts IS NOT NULL
       AND DATEDIFF(next_admit_ts, discharge_ts) BETWEEN 0 AND 30
      THEN 1 ELSE 0
    END AS is_readmission_30d
  FROM sequenced
)
SELECT
  COUNT(*) AS total_discharges,
  SUM(is_readmission_30d)AS readmissions_30d,
  ROUND(100.0 * SUM(is_readmission_30d) / COUNT(*), 2) AS readmission_rate_pct
FROM flagged;

In [0]:
-- Monthly readmission trend
WITH inpatient_stays AS (
  SELECT patient_id, encounter_id, start_ts AS admit_ts, stop_ts AS discharge_ts
  FROM health_lakehouse.gold.fact_encounters
  WHERE encounter_class = 'inpatient' AND stop_ts IS NOT NULL
),
sequenced AS (
  SELECT *,
    LEAD(admit_ts) OVER (PARTITION BY patient_id ORDER BY admit_ts) AS next_admit_ts
  FROM inpatient_stays
),
flagged AS (
  SELECT *,
    CASE WHEN next_admit_ts IS NOT NULL
          AND DATEDIFF(next_admit_ts, discharge_ts) BETWEEN 0 AND 30
         THEN 1 ELSE 0 END AS is_readmission_30d
  FROM sequenced
)
SELECT
  d.year,
  d.month,
  d.month_name,
  COUNT(*)                        AS discharges,
  SUM(f.is_readmission_30d)       AS readmissions,
  ROUND(100.0 * SUM(f.is_readmission_30d) / COUNT(*), 2) AS rate_pct
FROM flagged f
JOIN health_lakehouse.gold.dim_date d
  ON DATE(f.discharge_ts) = d.date_day
WHERE d.year >= 2015          -- focus on the well-populated recent decade
GROUP BY d.year, d.month, d.month_name
ORDER BY d.year, d.month;

In [0]:
-- Readmission rate by patient age band 
WITH inpatient_stays AS (
  SELECT patient_id, encounter_id, start_ts AS admit_ts, stop_ts AS discharge_ts
  FROM health_lakehouse.gold.fact_encounters
  WHERE encounter_class = 'inpatient' AND stop_ts IS NOT NULL
),
sequenced AS (
  SELECT *,
    LEAD(admit_ts) OVER (PARTITION BY patient_id ORDER BY admit_ts) AS next_admit_ts
  FROM inpatient_stays
),
flagged AS (
  SELECT *,
    CASE WHEN next_admit_ts IS NOT NULL
          AND DATEDIFF(next_admit_ts, discharge_ts) BETWEEN 0 AND 30
         THEN 1 ELSE 0 END AS is_readmission_30d
  FROM sequenced
)
SELECT
  p.age_band,
  COUNT(*)                        AS discharges,
  SUM(f.is_readmission_30d)       AS readmissions,
  ROUND(100.0 * SUM(f.is_readmission_30d) / COUNT(*), 2) AS rate_pct
FROM flagged f
JOIN health_lakehouse.gold.dim_patient p
  ON f.patient_id = p.patient_id
GROUP BY p.age_band
ORDER BY p.age_band;